# Sanity 2D — checkerboard (Lipman 2023, Fig. 4 gauche)

Avant de cramer 30h de A100 sur CIFAR-10, on valide la pipeline sur le toy 2D du papier : densité en damier, MLP 5×512. Les **trois modèles** (FM-OT, FM-Diff, DDPM) doivent reproduire le damier en quelques minutes de CPU/GPU local. Si l'effet "OT plus rapide" n'apparaît pas ici, il y a un bug — inutile de lancer le gros entraînement.

Cellule ~15 min CPU.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

import torch, math
import matplotlib.pyplot as plt
from torch.optim import Adam

from flow_matching_b3.paths import get_path
from flow_matching_b3.losses import get_loss_fn
from flow_matching_b3.sampling import sample, make_vf, euler_sample
from flow_matching_b3.unet import MLPVectorField

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Données — densité en damier

In [ ]:
def sample_checkerboard(n: int) -> torch.Tensor:
    """Sample n points from a 4x4 checkerboard pattern on [-2, 2]^2."""
    x = torch.rand(n) * 4 - 2
    y = torch.rand(n) * 4 - 2
    # Keep only the 'white' squares of a 4x4 checkerboard.
    keep = (torch.floor(x + 2).long() + torch.floor(y + 2).long()) % 2 == 0
    pts = torch.stack([x[keep], y[keep]], dim=-1)
    while pts.shape[0] < n:
        more = sample_checkerboard(n)
        pts = torch.cat([pts, more])[:n]
    return pts[:n]

data = sample_checkerboard(20_000)
plt.figure(figsize=(4, 4)); plt.scatter(data[:, 0], data[:, 1], s=1); plt.axis("equal"); plt.title("target — checkerboard"); plt.show()

## 2. Entraîner les trois variantes

Même MLP, même optim, même nombre de steps. Seul le `PATH_TYPE` change.

In [ ]:
def train_2d(path_name: str, steps: int = 5_000, batch: int = 512, lr: float = 1e-3) -> MLPVectorField:
    torch.manual_seed(0)
    model = MLPVectorField(dim=2).to(device)
    optim = Adam(model.parameters(), lr=lr)
    if path_name == "ot":
        path = get_path("ot", sigma_min=1e-4)
    elif path_name == "vp":
        path = get_path("vp")
    else:
        path = get_path("ddpm")
    loss_fn = get_loss_fn(path)
    data_full = data.to(device)
    for step in range(steps):
        idx = torch.randint(0, data_full.size(0), (batch,))
        x1 = data_full[idx]
        loss, _ = loss_fn(model, x1, path)
        optim.zero_grad(set_to_none=True); loss.backward(); optim.step()
        if step % 1_000 == 0:
            print(f"  [{path_name}] step {step:5d}  loss={loss.item():.4f}")
    return model, path

models = {}
for name in ["ot", "vp", "ddpm"]:
    print(f"=== training {name} ===")
    models[name] = train_2d(name, steps=5_000)

## 3. Comparer les samples finaux

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, (model, path)) in zip(axes, models.items()):
    model.eval()
    samples, _ = sample(model, path, shape=(5_000, 2), solver="dopri5", device=device, seed=42)
    samples = samples.cpu()
    ax.scatter(samples[:, 0], samples[:, 1], s=1)
    ax.set_xlim(-2.2, 2.2); ax.set_ylim(-2.2, 2.2); ax.set_aspect("equal")
    ax.set_title(f"{name}")
plt.suptitle("Samples (dopri5)"); plt.show()

## 4. Tracer les trajectoires (≈ Fig. 4 gauche du papier)

Pour chaque modèle, on prend les mêmes 800 points de bruit initial puis on les intègre par Euler avec un faible NFE, en sauvant la position à chaque pas.

In [ ]:
@torch.no_grad()
def integrate_traj(model, path, x0, nfe=20):
    vf = make_vf(model, path)
    dt = 1.0 / nfe
    traj = [x0.clone()]
    x = x0
    for k in range(nfe):
        t = torch.full((x.size(0),), k * dt, device=x.device)
        x = x + dt * vf(x, t)
        traj.append(x.clone())
    return torch.stack(traj)  # (nfe+1, N, 2)

torch.manual_seed(123)
x0_shared = torch.randn(800, 2, device=device)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, (model, path)) in zip(axes, models.items()):
    model.eval()
    traj = integrate_traj(model, path, x0_shared, nfe=30).cpu()
    for i in range(traj.shape[1]):
        ax.plot(traj[:, i, 0], traj[:, i, 1], lw=0.3, alpha=0.4)
    ax.scatter(traj[-1, :, 0], traj[-1, :, 1], s=2, color="red")
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5); ax.set_aspect("equal")
    ax.set_title(f"{name} — trajectories (Euler 30)")
plt.suptitle("Reproduction Fig. 4 (gauche) — OT ≈ lignes droites, diffusion = courbes"); plt.show()

## 5. Critère de validation

**À ce stade, on doit voir** :
- Les 3 modèles couvrent à peu près la densité en damier (samples à droite).
- Les trajectoires OT sont **visiblement plus droites** que VP/DDPM (cf. Fig. 4 du papier).

Si l'un de ces points échoue, **stop, débugger avant de lancer CIFAR-10**. Sinon → P3.